In [27]:
!pip install plotly

In [28]:
!pip install nbformat --upgrade

In [29]:
import pandas as pd 
from datetime import datetime as dt, timedelta
import plotly.express as px
import plotly.graph_objects as go 
import plotly.colors 

In [30]:
data=pd.read_csv("D:\Customer-Segmentation\online_retail_II.csv.zip")
data.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [31]:
data.tail()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France
1067370,581587,POST,POSTAGE,1,2011-12-09 12:50:00,18.00,12680.0,France


In [32]:
data.dropna(subset=['Customer ID'],inplace=True)

In [33]:
data['InvoiceDate']=pd.to_datetime(data['InvoiceDate'])
data['TotalAmount']=data['Quantity']*data['Price']

In [34]:
data.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalAmount
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [35]:
reference_date=pd.Timestamp(dt.now().date())

In [36]:
reference_date=data['InvoiceDate'].max()+ timedelta(days=1)

In [37]:
reference_date

Timestamp('2011-12-10 12:50:00')

In [38]:
rfm=data.groupby('Customer ID').agg({
    'InvoiceDate':lambda x:(reference_date - x.max()).days,
    'Invoice':'count',
    'TotalAmount':'sum'
})

In [39]:
rfm.rename(columns={'InvoiceDate':'Recency','Invoice':'Frequency','TotalAmount':'Monetary'},inplace=True)
rfm.head()

,Recency,Frequency,Monetary
Customer ID,,,
12346.0,326,48,-64.68
12347.0,2,253,5633.32
12348.0,75,51,2019.40
12349.0,19,180,4404.54
12350.0,310,17,334.40


In [40]:
#Define quantiles
quantiles= rfm.quantile(q=[0.25,0.50,0.75])

#Assign RFM Scores
def rfmscore(x,p,d):
    if p=='Recency':
        if x <= d[p] [0.25]:
            return 4
        elif x <= d[p] [0.50]:
            return 3
        elif x <= d[p] [0.75]:
            return 2
        else:
            return 1
    else:
        if x <= d[p] [0.25]:
            return 1
        elif x <= d[p] [0.50]:
            return 2
        elif x <= d[p] [0.75]:
            return 3
        else:
            return 4
        
rfm['R'] = rfm['Recency'].apply(rfmscore, args=('Recency',quantiles,))
rfm['F'] = rfm['Frequency'].apply(rfmscore, args=('Frequency',quantiles,))
rfm['M'] = rfm['Monetary'].apply(rfmscore, args=('Monetary',quantiles,))

        
rfm.head()

       

,Recency,Frequency,Monetary,R,F,M
Customer ID,,,,,,
12346.0,326,48,-64.68,2,2,1
12347.0,2,253,5633.32,4,4,4
12348.0,75,51,2019.40,3,2,3
12349.0,19,180,4404.54,4,4,4
12350.0,310,17,334.40,2,1,2


In [41]:
rfm['RFM_Segment'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)
rfm['RFM_Score'] = rfm[['R','F','M']].sum(axis=1)

In [42]:
rfm.head()

,Recency,Frequency,Monetary,R,F,M,RFM_Segment,RFM_Score
Customer ID,,,,,,,,
12346.0,326,48,-64.68,2,2,1,221,5
12347.0,2,253,5633.32,4,4,4,444,12
12348.0,75,51,2019.40,3,2,3,323,8
12349.0,19,180,4404.54,4,4,4,444,12
12350.0,310,17,334.40,2,1,2,212,5


In [43]:
Segment_labels=['Champion','Loyal','New customer','Mid-tire','At risk','Lost']
def assign_segment(row):
    score = row['RFM_Score']
    f_score = row['F']
    if score >=10:
        return 'Champion'
    elif 8 <= score >=10:
        return 'Loyal'
    elif 6 <= score <=8:
        if f_score == 1:
            return 'New customer'
        else: 
            return 'Mid-tire'
    elif 4 <= score <=6:
        return 'At risk'
    else:
        return'Lost'
rfm['RFM_Segment_label'] = rfm.apply(assign_segment, axis=1)
rfm.head()

,Recency,Frequency,Monetary,R,F,M,RFM_Segment,RFM_Score,RFM_Segment_label
Customer ID,,,,,,,,,
12346.0,326,48,-64.68,2,2,1,221,5,At risk
12347.0,2,253,5633.32,4,4,4,444,12,Champion
12348.0,75,51,2019.40,3,2,3,323,8,Mid-tire
12349.0,19,180,4404.54,4,4,4,444,12,Champion
12350.0,310,17,334.40,2,1,2,212,5,At risk


In [44]:
segment_counts = rfm['RFM_Segment_label'].value_counts().reset_index()
segment_counts.columns = ['RFM_Segment','Count']
segment_counts = segment_counts.sort_values('RFM_Segment')
segment_counts.sort_index()

,RFM_Segment,Count
0,Champion,1761
1,Mid-tire,1597
2,At risk,1170
3,Lost,1140
4,New customer,274


In [45]:
#Create the bsr chart using plotly
fig = px.bar(segment_counts,
             x='RFM_Segment',
             y='Count',
             title='Customer Distribution by RFM Segment',
             labels={'RFM_Segment':'RFM segment','Count':'Number of Customers'},
             color= 'RFM_Segment',
             color_discrete_sequence=px.colors.qualitative.Pastel
             )
fig.show()

In [46]:
champion_segment= rfm[rfm['RFM_Segment_label']=='Champion']


In [47]:
#Box plot for champion 
fig = go.Figure()
fig.add_trace(go.Box(y=champion_segment['Recency'], name='Recency'))
fig.add_trace(go.Box(y=champion_segment['Frequency'], name='Frequency'))
fig.add_trace(go.Box(y=champion_segment['Monetary'], name='Monetary'))


In [48]:
correlation_matrix = champion_segment[['R','F','M']].corr()
correlation_matrix

,R,F,M
R,1.000000,-0.198857,-0.208334
F,-0.198857,1.000000,0.260321
M,-0.208334,0.260321,1.000000


In [49]:
#create heatmap for champion correlation
fig_heatmap = go.Figure(data=go.Heatmap(
    z=correlation_matrix.values,
    x=correlation_matrix.columns,
    y=correlation_matrix.columns,
    colorscale='RdBu',
    colorbar=dict(title='Correlation')
))
fig_heatmap.update_layout(title='Corrlation Martix of RFM Values within Champions Segment')
fig_heatmap.show()

In [50]:
# bar plot
pastel_colors= plotly.colors.qualitative.Pastel
fig=go.Figure(data=[go.Bar(x=segment_counts.index, y=segment_counts.values,
                           marker=dict(color=pastel_colors))])
vip_color = 'rgb(158,202,225)'
fig.update_traces(marker_color=[vip_color if segment == 'Champions' else pastel_colors[i]
                                for i, segment in enumerate(segment_counts.index)],
                  marker_line_color='rgb(8,48,107)',
                  marker_line_width=1.5, opacity=0.6)
#Update the layout
fig.update_layout(title='Comparision of RFM Segments',
                  xaxis_title='RFM Segments',
                  yaxis_title='Number of Customers',
                  showlegend=False)
fig.show()

In [51]:
segment_scores = rfm.groupby('RFM_Segment_label')[['R','F','M']].mean().reset_index()
fig=go.Figure()

#Add bars for Recency score
fig.add_trace(go.Bar(
    x=segment_scores['RFM_Segment_label'],
    y=segment_scores['R'],
    name='Recency Score',
    marker_color='rgb(158,202,225)'
)) 

#Add bars for Frequency score
fig.add_trace(go.Bar(
    x=segment_scores['RFM_Segment_label'],
    y=segment_scores['F'],
    name='Frequency Score',
    marker_color='rgb(94,158,217)'
)) 

#Add bars for Monetary score
fig.add_trace(go.Bar(
    x=segment_scores['RFM_Segment_label'],
    y=segment_scores['M'],
    name='Monetary Score',
    marker_color='rgb(32,102,148)'
)) 

# Update the layout
fig.update_layout(
    title='Comparision of RFM Segments based on Recency, Frequency, and Monetary Scores',
    xaxis_title='RFM Segments',
    yaxis_title='Score',
    barmode='group',
    showlegend=True
)
fig.show()

In [52]:
rfm.to_csv('rfm_data.csv',index=False)